# Train the fretlift transcription model on Colab

1. **Runtime -> Change runtime type -> GPU** (a T4 is fine).
2. Run the cells in order. Data and runs are saved to Google Drive, so a
   disconnected session can pick up where it left off (`--resume`).

See `docs/TRAINING.md` in the repo for what each step does.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## Setup

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
WORK = "/content/drive/MyDrive/fretlift"   # data, cache and runs live here
!mkdir -p $WORK

In [ ]:
REPO = "https://github.com/jdinkha/fretlift.git"
!git clone -q $REPO /content/fretlift || (cd /content/fretlift && git pull -q)
%cd /content/fretlift
# Colab already has a CUDA build of PyTorch, so this only adds what's missing.
!pip install -q -e ".[train,dev]"
!pytest -q

## Step 1: data and the Basic Pitch baseline

GuitarSet is about 1.5 GB. The baseline needs Basic Pitch; if installing it
conflicts with Colab's packages, skip the baseline cells here and run them in
a local virtual environment instead.

In [ ]:
!fretlift-data guitarset --root $WORK/data/guitarset --download
!fretlift-data stats --manifest $WORK/data/guitarset/manifest.jsonl

In [ ]:
!pip install -q basic-pitch
COMMON = f"--manifest {WORK}/data/guitarset/manifest.jsonl --cache-dir {WORK}/cache"
!fretlift-eval --basic-pitch --split test --test-groups guitarset-player-05 {COMMON} --out {WORK}/reports/bp-clean.json
!fretlift-eval --basic-pitch --split test --test-groups guitarset-player-05 {COMMON} --distort high_gain --out {WORK}/reports/bp-high_gain.json
!fretlift-sweep --basic-pitch --split val --val-groups guitarset-player-04 {COMMON} --out {WORK}/reports/bp-sweep.json

## Step 3: train on clean audio

In [ ]:
COMMON = f"--manifest {WORK}/data/guitarset/manifest.jsonl --cache-dir {WORK}/cache"
SPLIT = "--val-groups guitarset-player-04 --test-groups guitarset-player-05"
!fretlift-train {COMMON} {SPLIT} --workers 2 --out {WORK}/runs/gs-clean

## Step 4: train with augmentation

Optional: upload cabinet impulse responses (WAV) to `$WORK/irs` and add
`--ir-dir {WORK}/irs`. Colab has 2 CPU cores, so augmented epochs are slower
(the CPU makes the audio, the GPU waits).

In [ ]:
!fretlift-train {COMMON} {SPLIT} --workers 2 --augment full --val-distort high_gain --out {WORK}/runs/gs-aug

In [ ]:
# If the session disconnected mid-run:
# !fretlift-train {COMMON} {SPLIT} --workers 2 --augment full --val-distort high_gain --out {WORK}/runs/gs-aug --resume {WORK}/runs/gs-aug/last.pt

## Compare

In [ ]:
import json, pathlib
for run in ["gs-clean", "gs-aug"]:
    p = pathlib.Path(WORK) / "runs" / run / "test_report.json"
    if p.exists():
        r = json.loads(p.read_text())
        for dist in [k for k in r if k != "decode"]:
            s = r[dist]["all"]
            print(f"{run:10s} {dist:10s} note F1 {s['note_f1']:.3f}  tab F1 {s['tab_f1'] or 0:.3f}")
for name in ["bp-clean", "bp-high_gain"]:
    p = pathlib.Path(WORK) / "reports" / f"{name}.json"
    if p.exists():
        s = json.loads(p.read_text())["summary"]["all"]
        print(f"{name:21s} note F1 {s['note_f1']:.3f}  tab F1 {s['tab_f1'] or 0:.3f}")

In [ ]:
import pandas as pd
pd.read_csv(f"{WORK}/runs/gs-aug/history.csv").plot(x="epoch", y=["clean_note_f1", "high_gain_note_f1", "clean_tab_f1"], grid=True)